In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv(r'C:\Users\Admin\Downloads\archive\Bengaluru_House_Data.csv')

# First look at the data
print("Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())
df.head()

Shape: (13320, 9)

Column Names: ['area_type', 'availability', 'location', 'size', 'society', 'total_sqft', 'bath', 'balcony', 'price']


,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [3]:
# Check missing values
print("Missing Values:\n", df.isnull().sum())
print("\nData Types:\n", df.dtypes)

Missing Values:
 area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64

Data Types:
 area_type        object
availability     object
location         object
size             object
society          object
total_sqft       object
bath            float64
balcony         float64
price           float64
dtype: object


In [12]:
# Fix total_sqft column - convert ranges like "1000-1500" to average
def convert_sqft(x):
    try:
        if '-' in str(x):
            parts = str(x).split('-')
            return (float(parts[0]) + float(parts[1])) / 2
        return float(x)
    except:
        return None

df['total_sqft'] = df['total_sqft'].apply(convert_sqft)

# Drop rows where conversion failed
df = df.dropna(subset=['total_sqft'])

# Drop that 1 missing location row
df = df.dropna(subset=['location'])

# Clean location column
df['location'] = df['location'].apply(lambda x: x.strip())

# Add price_per_sqft column
df['price_per_sqft'] = df['price'] * 100000 / df['total_sqft']

# Remove outliers
df = df[df['price_per_sqft'] > 300]
df = df[df['price_per_sqft'] < 50000]

# Remove bath outliers
df = df[df['bath'] < df['bhk'] + 2]

print("Final Shape:", df.shape)
df.head()

Final Shape: (13068, 9)


,area_type,availability,location,total_sqft,bath,balcony,price,bhk,price_per_sqft
0,Super built-up Area,19-Dec,Electronic City Phase II,1056.0,2.0,1.0,39.07,2,3699.810606
1,Plot Area,Ready To Move,Chikka Tirupathi,2600.0,5.0,3.0,120.00,4,4615.384615
2,Built-up Area,Ready To Move,Uttarahalli,1440.0,2.0,3.0,62.00,3,4305.555556
3,Super built-up Area,Ready To Move,Lingadheeranahalli,1521.0,3.0,1.0,95.00,3,6245.890861
4,Super built-up Area,Ready To Move,Kothanur,1200.0,2.0,1.0,51.00,2,4250.000000


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

# Encode categorical columns
le = LabelEncoder()
df['location_enc'] = le.fit_transform(df['location'])
df['area_type_enc'] = le.fit_transform(df['area_type'])

# Select features and target
X = df[['total_sqft', 'bath', 'balcony', 'bhk', 'price_per_sqft', 'location_enc', 'area_type_enc']]
y = df['price']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train 3 models
lr = LinearRegression()
dt = DecisionTreeRegressor()
rf = RandomForestRegressor()

lr.fit(X_train, y_train)
dt.fit(X_train, y_train)
rf.fit(X_train, y_train)

# Evaluate
print("Linear Regression R² :", round(r2_score(y_test, lr.predict(X_test)), 3))
print("Decision Tree R²     :", round(r2_score(y_test, dt.predict(X_test)), 3))
print("Random Forest R²     :", round(r2_score(y_test, rf.predict(X_test)), 3))

Linear Regression R² : 0.783
Decision Tree R²     : 0.973
Random Forest R²     : 0.987


In [14]:
# Detailed evaluation of best model (Random Forest)
y_pred = rf.predict(X_test)

print("=== Random Forest Results ===")
print("R² Score  :", round(r2_score(y_test, y_pred), 3))
print("MAE       :", round(mean_absolute_error(y_test, y_pred), 2), "lakhs")

# Predict a sample house
sample = [[1500, 2, 1, 3, 5000, 100, 1]]
predicted_price = rf.predict(sample)
print("\n🏠 Sample Prediction:")
print("3 BHK, 1500 sqft → ₹", round(predicted_price[0], 2), "lakhs")

=== Random Forest Results ===
R² Score  : 0.987
MAE       : 2.16 lakhs

🏠 Sample Prediction:
3 BHK, 1500 sqft → ₹ 74.95 lakhs


c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [15]:
import pickle

# Save the model
with open('house_price_model.pkl', 'wb') as f:
    pickle.dump(rf, f)

print("✅ Model saved as house_price_model.pkl")

✅ Model saved as house_price_model.pkl
